### Глобальная модель (multi-resist) — `16_model_comparison_global_multi_resist_mlforecast` (MLForecast)

Тот же конвейер данных, что в [`15_model_comparison_global_multi_resist_boost.ipynb`](15_model_comparison_global_multi_resist_boost.ipynb), но **таргет-лаги и скользящие по `fact_value`** строит **`MLForecast`** (`lags`, `lag_transforms`, `date_features`). Экзогенные **`calc_*`** — по-прежнему сдвиги (`add_calc_shift_lags`) и full flatten (`add_calc_history_flatten`).

**Отличие от `15` (оценка):** в `15` тест предсказывается **построчно (direct)** с признаками, где лаги по факту считаны из **фактической истории**, в том числе по месяцам теста (oracle-лаги). Здесь прогноз на июль–декабрь строится **`predict(h=6)`** — **рекурсивно (recursive forecasting)**: на каждом шаге в ряд подставляется прогноз, признаки пересчитываются. Метрики на том же окне **июль–декабрь 2025**, но **не сопоставимы напрямую** с «Итог» в `15` без этой оговорки.

**On-premise (on-premise):** `mlforecast` — локальная библиотека Python, внешние API не требуются.

Модели: LGBM (2), XGB (2), DT-deep, DT-ccp, RF-default, GBR-default (без CatBoost в этой версии). Тест: июль–декабрь 2025.


In [ ]:
import json
import sys
import time
import copy
from pathlib import Path

import importlib

_ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import scripts.calc_lag_features as _calc_lag_mod
import scripts.calc_history_flatten as _calc_hist_mod

importlib.reload(_calc_lag_mod)
importlib.reload(_calc_hist_mod)

from scripts.calc_lag_features import (
    add_calc_shift_lags,
    calc_lag_columns_for_sets,
    DEFAULT_CALC_SHIFTS,
)
from scripts.calc_history_flatten import add_calc_history_flatten

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, ExponentiallyWeightedMean

%matplotlib inline
import warnings

warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)


#### Конфигурация

In [ ]:
RESIST_ID = 124
RESIST_IDS = [124]
GROUP_COLS = ("resist_id", "operation_id")
SUMMARY_TOP = 15

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]
HORIZON = len(TEST_MONTHS)

CALC_WHITELIST = [
    "calc_workDayMonth_2",
    "calc_hourInWeek_12",
    "calc_numberWorkHour_1",
    "calc_effective_13",
    "calc_absenteeism_14",
    "calc_employFact_7",
    "calc_resist_10",
]

OUT_XLSX = Path("../data/processed/model_comparison_global_multi_resist_mlforecast.xlsx")

STATIC_FEATURES = ["resist_code", "op_code"]
BASE_DYNAMIC = ["workDayMonth_2", "calculate_minute"]


#### Загрузка данных и подготовка (без ручных лагов по `fact_value`)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"Файлы данных не найдены в {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print(f"calc_* колонки ({len(calc_cols)}):", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

RESIST_IDS = list(dict.fromkeys(RESIST_IDS))
avail = set(int(x) for x in df_model39["resist_id"].dropna().unique())
missing = [r for r in RESIST_IDS if r not in avail]
if missing:
    sample = sorted(avail)[:15]
    raise ValueError(
        f"Нет данных для resist_id={missing}. Примеры доступных id: {sample}..."
    )
print(f"RESIST_IDS ({len(RESIST_IDS)}): {RESIST_IDS}")


#### Метрики

In [ ]:
from typing import Union


def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0,
            100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo


def _clip_predictions(s: Union[pd.Series, np.ndarray]) -> Union[pd.Series, np.ndarray]:
    """MLForecast может передать в callback и Series, и ndarray."""
    if isinstance(s, pd.Series):
        return pd.Series(
            np.maximum(s.to_numpy(dtype=float), 0.0),
            index=s.index,
            name=s.name,
        )
    return np.maximum(np.asarray(s, dtype=float), 0.0)


#### MLForecast: long format, `fit` на train, рекурсивный `predict(h)` на тесте

In [ ]:
def _wide_to_long(wide: pd.DataFrame, dynamic_cols: list[str]) -> pd.DataFrame:
    w = wide.copy()
    w["ds"] = pd.to_datetime(dict(year=w["year"], month=w["month"], day=1), errors="coerce")
    w["unique_id"] = w["resist_id"].astype(str) + "__" + w["operation_id"].astype(str)
    w["y"] = w["fact_value"]
    use = ["unique_id", "ds", "y"] + STATIC_FEATURES + dynamic_cols
    for c in dynamic_cols:
        w[c] = pd.to_numeric(w[c], errors="coerce").fillna(0.0)
    for c in STATIC_FEATURES:
        w[c] = pd.to_numeric(w[c], errors="coerce").fillna(0.0)
    return w[use].dropna(subset=["ds", "y"]).sort_values(["unique_id", "ds"]).reset_index(drop=True)


def _build_mlf_models() -> dict[str, object]:
    return {
        "LGBM-default": LGBMRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.1,
            verbose=-1,
            random_state=42,
        ),
        "LGBM-tuned": LGBMRegressor(
            n_estimators=800,
            max_depth=8,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=10,
            verbose=-1,
            random_state=42,
        ),
        "XGB-default": XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.1,
            verbosity=0,
            random_state=42,
        ),
        "XGB-regularized": XGBRegressor(
            n_estimators=800,
            max_depth=8,
            learning_rate=0.05,
            reg_alpha=0.1,
            reg_lambda=5,
            verbosity=0,
            random_state=42,
        ),
        "DT-deep": DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=42),
        "DT-ccp": DecisionTreeRegressor(
            max_depth=15,
            min_samples_leaf=2,
            ccp_alpha=0.02,
            random_state=42,
        ),
        "RF-default": RandomForestRegressor(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
        "GBR-default": GradientBoostingRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
        ),
    }


# Без add_fact_lag_columns: лаги целевого ряда задаёт MLForecast
df = df_model39[df_model39["resist_id"].isin(RESIST_IDS)].copy()
df = df.sort_values(list(GROUP_COLS) + ["year", "month"], kind="mergesort").reset_index(drop=True)
df["op_code"] = df["operation_id"].astype("category").cat.codes
df["resist_code"] = df["resist_id"].astype("category").cat.codes

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0)

df_shift, _lag_calc_cols, calc_keys_used = add_calc_shift_lags(
    df.copy(), drop_scalar_calc=True, group_cols=GROUP_COLS
)
b_flat, c_flat = calc_lag_columns_for_sets(calc_keys_used, whitelist=CALC_WHITELIST)
print(
    f"calc_* lag-сдвиги: ключей {len(calc_keys_used)}; "
    f"набор B: {len(b_flat)}, набор C: {len(c_flat)}; shift-месяцы: {DEFAULT_CALC_SHIFTS}"
)

df_flat, flatten_calc_cols, calc_keys_hist = add_calc_history_flatten(
    df.copy(), drop_scalar_calc=True, merge_keys=GROUP_COLS
)
b_hist_flat = flatten_calc_cols
print(
    f"full history flatten: ключей calc {len(calc_keys_hist)}, "
    f"колонок calc_k__YYYY_MM {len(flatten_calc_cols)}"
)

FEATURE_SETS: dict[str, list[str]] = {
    "B+lags": BASE_DYNAMIC + b_flat,
    "C+lags": BASE_DYNAMIC + c_flat,
    "B_hist+lags": BASE_DYNAMIC + b_hist_flat,
}
FEATURE_DF = {
    "B+lags": df_shift,
    "C+lags": df_shift,
    "B_hist+lags": df_flat,
}

for fs_name, fs_cols in FEATURE_SETS.items():
    dfx = FEATURE_DF[fs_name]
    miss = [c for c in fs_cols if c not in dfx.columns]
    if miss:
        raise ValueError(f"Не хватает колонок для {fs_name}: {miss[:20]}...")
    print(f"  {fs_name}: динамических экзогенных {len(fs_cols)} (+ static {STATIC_FEATURES})")

mask_test = (df_shift["year"] == TEST_YEAR) & (df_shift["month"].isin(TEST_MONTHS))
mask_train = ~mask_test
n_test = int(mask_test.sum())
n_train = int(mask_train.sum())
print(f"Train: {n_train} строк, Test: {n_test} строк")

y_test = df_shift.loc[mask_test, "fact_value"].values
t_fp, s_fp, q_fp = calc_metrics(y_test, df_shift.loc[mask_test, "fact_predict"].values)
print(
    f"Baseline fact_predict: Точность={t_fp}%, Стабильность={s_fp}%, Качество={q_fp}%"
)

if n_train == 0:
    raise RuntimeError("Train пуст — нельзя обучить модели.")

LAG_TRANSFORMS = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        ExponentiallyWeightedMean(alpha=0.3),
    ],
}
DATE_FEATURES = ["month", "quarter", "year"]
LAGS_MLF = [1, 2, 3, 6, 12]

results: list = []
preds_store: dict[tuple[str, str], np.ndarray] = {}
model_configs = _build_mlf_models()

df_test_meta = df_shift.loc[mask_test, ["resist_id", "operation_id", "year", "month"]].copy()
df_test_meta["unique_id"] = (
    df_test_meta["resist_id"].astype(str) + "__" + df_test_meta["operation_id"].astype(str)
)
df_test_meta["ds"] = pd.to_datetime(
    dict(year=df_test_meta["year"], month=df_test_meta["month"], day=1), errors="coerce"
)
order_keys = df_test_meta[["unique_id", "ds"]].copy()

for fs_name, dyn_cols in FEATURE_SETS.items():
    wide = FEATURE_DF[fs_name]
    long_train = _wide_to_long(wide.loc[mask_train], dyn_cols)
    if long_train.empty:
        print(f"Пропуск {fs_name}: пустой train в long format")
        continue

    sub_te = wide.loc[mask_test].copy()
    exog_test = sub_te[dyn_cols].copy()
    for c in dyn_cols:
        exog_test[c] = pd.to_numeric(exog_test[c], errors="coerce").fillna(0.0)
    exog_test["unique_id"] = (
        sub_te["resist_id"].astype(str) + "__" + sub_te["operation_id"].astype(str)
    )
    exog_test["ds"] = pd.to_datetime(
        dict(year=sub_te["year"], month=sub_te["month"], day=1), errors="coerce"
    )

    for config_name, model in model_configs.items():
        fcst = MLForecast(
            models=copy.deepcopy(model),
            freq="MS",
            lags=LAGS_MLF,
            lag_transforms=LAG_TRANSFORMS,
            date_features=DATE_FEATURES,
            num_threads=-1,
        )
        fcst.fit(long_train, static_features=STATIC_FEATURES)
        fd = fcst.make_future_dataframe(h=HORIZON)
        X_df = fd.merge(
            exog_test[["unique_id", "ds"] + dyn_cols],
            on=["unique_id", "ds"],
            how="left",
        )
        X_df[dyn_cols] = X_df[dyn_cols].fillna(0.0)
        t0 = time.perf_counter()
        preds = fcst.predict(
            h=HORIZON,
            X_df=X_df,
            after_predict_callback=_clip_predictions,
        )
        elapsed = round(time.perf_counter() - t0, 2)
        pred_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
        merged = order_keys.merge(
            preds[["unique_id", "ds", pred_col]],
            on=["unique_id", "ds"],
            how="left",
        )
        y_pred = np.maximum(np.nan_to_num(merged[pred_col].to_numpy(dtype=float), nan=0.0), 0.0)
        combo_key = (config_name, fs_name)
        preds_store[combo_key] = y_pred
        tochnost, stab, kachestvo = calc_metrics(y_test, y_pred)
        results.append(
            {
                "Конфигурация": config_name,
                "Набор признаков": fs_name,
                "Тип": "MLForecast (global, recursive)",
                "Точность, %": tochnost,
                "Стабильность, %": stab,
                "Качество, %": kachestvo,
                "Время, сек": elapsed,
                "Точн. fact_predict, %": t_fp,
                "Стаб. fact_predict, %": s_fp,
                "Кач. fact_predict, %": q_fp,
            }
        )
        print(f"{config_name} | {fs_name}: Качество={kachestvo}% ({elapsed}s)")

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Качество, %", ascending=False).reset_index(drop=True)
print("\n--- Топ по Качеству ---")
print(df_results.head(SUMMARY_TOP).to_string(index=False))

df_test = df_shift.loc[mask_test]
per_rid_rows: list = []
for rid in RESIST_IDS:
    sub = df_test["resist_id"].values == rid
    if not sub.any():
        continue
    yt = df_test["fact_value"].values[sub]
    for (config_name, fs_name), y_pred in preds_store.items():
        yp = y_pred[sub]
        t, s, k = calc_metrics(yt, yp)
        per_rid_rows.append(
            {
                "resist_id": rid,
                "Конфигурация": config_name,
                "Набор признаков": fs_name,
                "Точность, %": t,
                "Стабильность, %": s,
                "Качество, %": k,
            }
        )
df_per_rid = pd.DataFrame(per_rid_rows)

MLF_PARAMS = {
    "freq": "MS",
    "lags": LAGS_MLF,
    "lag_transforms": str(LAG_TRANSFORMS),
    "date_features": DATE_FEATURES,
    "horizon": HORIZON,
}
df_params = pd.DataFrame([MLF_PARAMS])

OUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    df_results.to_excel(w, sheet_name="Итог", index=False)
    df_per_rid.to_excel(w, sheet_name="По resist_id", index=False)
    df_params.to_excel(w, sheet_name="MLForecast_params", index=False)
print(f"\nСохранено: {OUT_XLSX.resolve()}")
